In [1]:
pip install folium

Note: you may need to restart the kernel to use updated packages.


pip install selenium webdriver-manager

https://map.naver.com/p/api/search/all?query=종로 헬스장&type=all

In [2]:
import requests
import pandas as pd

def naver_place_search_requests(query, display=5):
    url = f"https://map.naver.com/p/api/search/all?query={query}&type=all"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Referer": "https://map.naver.com/v5/",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "Connection": "keep-alive"
    }

    res = requests.get(url, headers=headers)
    if res.status_code == 403:
        print("⚠️ 네이버에서 403 차단 발생")
        return pd.DataFrame()

    data = res.json()
    results = []
    for item in data.get("result", {}).get("place", {}).get("list", []):
        results.append({
            "이름": item.get("name"),
            "주소": item.get("address"),
            "평점": float(item.get("bizrntScore", 0)),
            "사진": item.get("thumUrl"),
            "경도": float(item.get("x", 0)),
            "위도": float(item.get("y", 0))
        })
    return pd.DataFrame(results).head(display)

# ✅ 사용 예시
df = naver_place_search_requests("종로구 헬스장", display=5)
print(df)


Empty DataFrame
Columns: []
Index: []


In [3]:
pip install selenium webdriver-manager selenium-wire


Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip uninstall -y blinker selenium-wire
!pip install "blinker<2.0" selenium-wire==5.1.0


Found existing installation: blinker 1.6.2
Uninstalling blinker-1.6.2:
  Successfully uninstalled blinker-1.6.2
Found existing installation: selenium-wire 5.1.0
Uninstalling selenium-wire-5.1.0:
  Successfully uninstalled selenium-wire-5.1.0
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached selenium_wire-5.1.0-py3-none-any.whl.metadata (49 kB)
Using cached selenium_wire-5.1.0-py3-none-any.whl (239 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)

   -------------------- ------------------- 1/2 [selenium-wire]
   -------------------- ------------------- 1/2 [selenium-wire]
   ---------------------------------------- 2/2 [selenium-wire]



In [5]:
!pip uninstall -y blinker selenium-wire
!pip install blinker==1.6.2 selenium-wire==5.1.0


Found existing installation: blinker 1.9.0
Uninstalling blinker-1.9.0:
  Successfully uninstalled blinker-1.9.0
Found existing installation: selenium-wire 5.1.0
Uninstalling selenium-wire-5.1.0:
  Successfully uninstalled selenium-wire-5.1.0
  Using cached blinker-1.6.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached selenium_wire-5.1.0-py3-none-any.whl.metadata (49 kB)
Using cached blinker-1.6.2-py3-none-any.whl (13 kB)
Using cached selenium_wire-5.1.0-py3-none-any.whl (239 kB)

   -------------------- ------------------- 1/2 [selenium-wire]
   -------------------- ------------------- 1/2 [selenium-wire]
   ---------------------------------------- 2/2 [selenium-wire]



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
flask 3.1.0 requires blinker>=1.9, but you have blinker 1.6.2 which is incompatible.


In [6]:
import gzip
import json
import pandas as pd
from seleniumwire import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from ipyleaflet import Map, Marker, Popup, TileLayer
from ipywidgets import HTML
import time

def naver_place_search_selenium(query, display=10, delay=5):
    """네이버 지도 검색 후 JSON 응답 캡처 → DataFrame"""
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    driver.get(f"https://map.naver.com/v5/search/{query}")
    time.sleep(delay)

    results = []
    for request in driver.requests:
        if "api/search/all" in request.url and request.response:
            try:
                body = request.response.body
                try:
                    body = gzip.decompress(body)
                except OSError:
                    pass  # 이미 해제됨

                data = json.loads(body.decode("utf-8"))

                for item in data.get("result", {}).get("place", {}).get("list", []):
                    results.append({
                        "이름": item.get("name"),
                        "주소": item.get("address"),
                        "평점": float(item.get("bizrntScore", 0)),
                        "사진": item.get("thumUrl"),
                        "경도": float(item.get("x", 0)),
                        "위도": float(item.get("y", 0))
                    })
            except Exception as e:
                print("JSON 파싱 실패:", e)

    driver.quit()
    return pd.DataFrame(results).head(display)


def recommend_top5(user_location):
    """위치 주변 헬스장/공원/병원 검색 후 평점순 TOP5"""
    categories = ["헬스장", "공원", "병원"]
    all_places = []

    for cat in categories:
        query = f"{user_location} {cat}"
        df = naver_place_search_selenium(query, display=10)
        if not df.empty:
            df["카테고리"] = cat
            all_places.append(df)

    if not all_places:
        return pd.DataFrame()

    df_all = pd.concat(all_places, ignore_index=True)
    return df_all.sort_values(by="평점", ascending=False).head(5).reset_index(drop=True)


# ✅ 사용 예시
user_location = "종로구"
df = recommend_top5(user_location)
print(df)

# ✅ 지도 표시
if not df.empty:
    m = Map(center=(df["위도"].mean(), df["경도"].mean()), zoom=13, scroll_wheel_zoom=True)
    tiles = TileLayer(url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
                      attribution="© OpenStreetMap contributors")
    m.add_layer(tiles)

    for row in df.itertuples():
        popup_html = HTML(f"""
            <b>{row.이름}</b><br>
            카테고리: {row.카테고리}<br>
            평점: {row.평점}<br>
            주소: {row.주소}<br>
            <img src="{row.사진}" width="150"><br>
        """)
        mk = Marker(location=(row.위도, row.경도), title=row.이름)
        mk.popup = Popup(child=popup_html, close_button=True)
        m.add_layer(mk)

    display(m)
else:
    print("⚠️ 검색 결과가 없습니다.")


                이름                                              주소   평점  \
0       에이치짐 신설동역점                서울특별시 종로구 숭인동 766 종로청계힐스테이트 지하1층  0.0   
1         스포짐 광화문점  서울특별시 종로구 내수동 72 경희궁의아침 3단지 오피스텔 지하2층 스포짐 피트니스  0.0   
2       MCTGYM 광화문       서울특별시 종로구 내수동 110-39 지하1층 광화문 헬스장 MCT GYM  0.0   
3        스포애니 종각역점                     서울특별시 종로구 종로2가 84-8 B1, B2층  0.0   
4  디팰리스 스포츠클럽 광화문점              서울특별시 종로구 신문로2가 171 B2층 디팰리스 스포츠클럽  0.0   

                                                  사진          경도         위도  \
0  https://ldb-phinf.pstatic.net/20250922_161/175...  127.021444  37.575547   
1  https://ldb-phinf.pstatic.net/20250919_12/1758...  126.972512  37.573226   
2  https://ldb-phinf.pstatic.net/20250710_138/175...  126.970987  37.572679   
3  https://ldb-phinf.pstatic.net/20250918_206/175...  126.984945  37.569906   
4  https://ldb-phinf.pstatic.net/20201208_215/160...  126.972015  37.568678   

  카테고리  
0  헬스장  
1  헬스장  
2  헬스장  
3  헬스장  
4  헬스장  


Map(center=[np.float64(37.57200726), np.float64(126.9843807)], controls=(ZoomControl(options=['position', 'zoo…

In [7]:
import folium
import pandas as pd
from ipywidgets import interact, Dropdown

# df = recommend_top5("종로구")  # 이미 얻은 데이터 사용

icon_urls = {
    "헬스장": "https://cdn-icons-png.flaticon.com/512/2964/2964514.png",
    "공원": "https://cdn-icons-png.flaticon.com/512/854/854878.png",
    "병원": "https://cdn-icons-png.flaticon.com/512/2966/2966327.png"
}

def make_map(category="전체"):
    m = folium.Map(location=[df["위도"].mean(), df["경도"].mean()], zoom_start=13)

    for row in df.itertuples():
        if category == "전체" or row.카테고리 == category:
            icon_url = icon_urls.get(row.카테고리)
            icon = folium.CustomIcon(icon_url, icon_size=(35, 35))
            popup = folium.Popup(
                f"<b>{row.이름}</b><br>"
                f"카테고리: {row.카테고리}<br>"
                f"평점: {row.평점}<br>"
                f"주소: {row.주소}<br>"
                f"<img src='{row.사진}' width='150'>",
                max_width=250
            )
            folium.Marker([row.위도, row.경도], icon=icon, popup=popup).add_to(m)

    return m

# 인터랙티브 UI
interact(
    make_map,
    category=Dropdown(options=["전체", "헬스장", "공원", "병원"], value="전체", description="보기:")
)


interactive(children=(Dropdown(description='보기:', options=('전체', '헬스장', '공원', '병원'), value='전체'), Output()), _…

<function __main__.make_map(category='전체')>